# 第 4 周：回归任务与验证流程

**对应主线**：为李宏毅课程 HW1 Regression 建立原创前置能力，不包含或改写官方作业内容。  
**任务**：根据三个合成环境特征预测每日能耗。  
**先修**：训练循环、MSE、NumPy shape；需要 `numpy`、`torch`。

本周重点不是“把训练 loss 压到最低”，而是：

1. 固定训练/验证划分；
2. 只用训练集统计量标准化；
3. 与简单基线比较；
4. 同时报告 MSE 和 MAE；
5. 保存一次可解释的实验结论。


In [ ]:
import numpy as np
import torch
from torch import nn

np.random.seed(11)
torch.manual_seed(11)


## 1. 创建原创合成数据


In [ ]:
rng = np.random.default_rng(11)
n = 360
temperature = rng.uniform(5, 35, n)
occupancy = rng.uniform(0, 1, n)
equipment_hours = rng.uniform(0, 12, n)

features_np = np.column_stack([temperature, occupancy, equipment_hours]).astype("float32")
target_np = (
    18
    + 0.7 * temperature
    + 24 * occupancy
    + 2.4 * equipment_hours
    + rng.normal(0, 2.0, n)
).astype("float32")

print(features_np.shape, target_np.shape)


## 2. 划分后再标准化

数据泄漏示例：用全部数据（包括验证集）计算均值和标准差。正确做法是只用训练集拟合统计量，再应用到验证集。


In [ ]:
indices = rng.permutation(n)
train_indices = indices[:288]
val_indices = indices[288:]

x_train_np = features_np[train_indices]
y_train_np = target_np[train_indices]
x_val_np = features_np[val_indices]
y_val_np = target_np[val_indices]

mean = x_train_np.mean(axis=0)
std = x_train_np.std(axis=0)
x_train_np = (x_train_np - mean) / std
x_val_np = (x_val_np - mean) / std

x_train = torch.tensor(x_train_np)
y_train = torch.tensor(y_train_np).reshape(-1, 1)
x_val = torch.tensor(x_val_np)
y_val = torch.tensor(y_val_np).reshape(-1, 1)


In [ ]:
assert x_train.shape == (288, 3)
assert x_val.shape == (72, 3)
assert set(train_indices).isdisjoint(set(val_indices))
assert np.allclose(x_train_np.mean(axis=0), 0, atol=1e-6)
print("划分与标准化测试通过 ✅")


## 3. 先建立基线

基线永远预测训练集标签均值。模型若不优于它，复杂训练没有价值。


In [ ]:
baseline_prediction = y_train.mean()
baseline_mse = ((y_val - baseline_prediction) ** 2).mean().item()
baseline_mae = (y_val - baseline_prediction).abs().mean().item()
print(f"baseline MSE={baseline_mse:.3f}, MAE={baseline_mae:.3f}")


## 4. 训练线性回归


In [ ]:
model = nn.Linear(3, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.2)

history = []
for epoch in range(1000):
    model.train()
    optimizer.zero_grad()
    train_prediction = model(x_train)
    train_loss = loss_fn(train_prediction, y_train)
    train_loss.backward()
    optimizer.step()

    if epoch % 100 == 0 or epoch == 999:
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(x_val), y_val).item()
        history.append((epoch, train_loss.item(), val_loss))

with torch.no_grad():
    val_prediction = model(x_val)
    val_mse = loss_fn(val_prediction, y_val).item()
    val_mae = (val_prediction - y_val).abs().mean().item()

print("最后三个检查点:", history[-3:])
print(f"model MSE={val_mse:.3f}, MAE={val_mae:.3f}")


In [ ]:
assert val_prediction.shape == y_val.shape
assert val_mse < baseline_mse * 0.15
assert val_mae < baseline_mae * 0.45
print("回归流程测试通过 ✅")


## 5. 实验解释练习

用自己的话补全，不要只复制数字：

- **输入**：每个样本有 ___ 个特征，分别代表 ___。
- **输出**：模型预测一个连续值，shape 是 ___。
- **损失**：训练使用 ___，因为 ___。
- **验证结果**：模型相对均值基线改善了 ___。
- **仍可能失败**：真实数据若出现 ___，本模型可能泛化较差。


In [ ]:
improvement = 1 - val_mse / baseline_mse
print(f"相对 MSE 改善：{improvement:.1%}")


## 常见错误

- 验证集参与训练或标准化统计量计算；
- 训练集和验证集使用不同的标准化规则；
- 只报告训练 loss；
- 不与基线比较；
- 把 MSE 的单位当成原目标单位（MAE 才与目标同单位）。

## 扩展

加入一个无用随机特征，比较验证误差和权重；再尝试故意减少训练样本，观察泛化变化。

**通过条件**：全部测试通过，并完成上面的五句实验解释。
